# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane (Refresh / Content Opportunity Scoring) is fundamentally a ranking/scoring problem, not plain classification. The starter pipeline trains a classifier under the hood (predicting is_declining_label), but what actually gets used is the ordered queue it produces -- a reviewer doesn't care about a probability number, they care about "who do I look at first." The metric that matters is precision@K (are the top K pages actually worth reviewing), not accuracy -- accuracy would treat a page ranked #1 and a page ranked #400 the same as long as both got the "declining" label right.

In [1]:
!git clone https://github.com/SuryaK5125/flyrank-ml.git
%cd flyrank-ml

import pandas as pd
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print("Task type check -- ranking, not plain classification")
print(f"Total pages: {len(df)}")
print(f"Pages flagged 'down': {(df['trend_direction']=='down').sum()} -- too many to review all at once, so ORDER matters, not just yes/no")

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 107, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 107 (delta 26), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (107/107), 1.84 MiB | 8.62 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/flyrank-ml
Task type check -- ranking, not plain classification
Total pages: 30000
Pages flagged 'down': 16262 -- too many to review all at once, so ORDER matters, not just yes/no


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Target or proxy

Target/proxy: is_declining_label, defined as trend_direction == "down". This is a DEFINED-RULE proxy, not a true observed future outcome -- trend_direction itself is computed from trend_pct over the current 90-day window, not measured after a decision point. I'm using it because it's what the starter pipeline provides and it's good enough to prove the workflow, but a stronger capstone target would be a future-window label (e.g. features from days 1-90 predicting decline in days 91-120) so the label reflects something that hasn't happened yet at prediction time.

In [3]:
print(df['trend_direction'].value_counts())
print()
print(f"Proxy label rate (trend_direction == 'down'): {(df['trend_direction']=='down').mean():.1%}")
# trend_direction and trend_pct are NEVER usable as features -- they define the label itself

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Proxy label rate (trend_direction == 'down'): 54.2%


## 3. Success metric

Success metric: Precision@50 -- of the top 50 pages the ranking puts first, how many are actually worth reviewing. This matches the real decision (a reviewer has fixed weekly capacity, not unlimited time) better than accuracy or ROC-AUC would, since those treat every page equally instead of caring about the top of the list. The starter pipeline's own numbers give me a number to beat: baseline rule Precision@50 = 0.240, random forest Precision@50 = 0.740 (from outputs/model_report.md). "Good" for my capstone means beating 0.240 by a clear, honest margin -- not just matching the existing model.

In [4]:
# Precision@50 in plain terms: of the top 50 ranked pages, how many are true positives?
# baseline rule vs random forest, from outputs/model_report.md
baseline_p50 = 0.240
rf_p50 = 0.740
print(f"Baseline rule catches ~{baseline_p50*50:.0f} of top 50 correctly")
print(f"Random forest catches ~{rf_p50*50:.0f} of top 50 correctly")
print(f"That's the bar my capstone needs to understand and try to beat honestly.")

Baseline rule catches ~12 of top 50 correctly
Random forest catches ~37 of top 50 correctly
That's the bar my capstone needs to understand and try to beat honestly.


## 4. The unit of analysis, as a real dataframe

Unit of analysis: one row = one content page (content_id), trailing-90-day snapshot. Below is a real slice of the columns that matter for this lane.

In [5]:
unit_of_analysis = df[['content_id', 'client_id', 'trend_direction', 'impressions_90d',
                        'days_since_last_update', 'avg_position', 'content_age_days']]
print(f"Shape: {unit_of_analysis.shape} -- one row per content page")
unit_of_analysis.head(10)

Shape: (30000, 7) -- one row per content page


,content_id,client_id,trend_direction,impressions_90d,days_since_last_update,avg_position,content_age_days
0,content_304f48230142,client_f369cb89fc,down,3803,20,10.6,187
1,content_a1fb4e703a9e,client_4e07408562,down,15320,25,20.3,445
2,content_9aa793d4d895,client_7f2253d7e2,down,12581,20,36.5,141
3,content_331d6c4de07b,client_19581e27de,stable,11751,22,6.2,463
4,content_d99b7a2d90ca,client_3fdba35f04,down,19140,14,44.0,263
5,content_d4084a4bc775,client_f369cb89fc,down,3970,20,8.5,147
6,content_9a34b442b552,client_8722616204,down,20,20,7.0,90
7,content_a63219c6e95a,client_19581e27de,stable,1724,22,21.2,445
8,content_5e6c160719bc,client_6208ef0f77,down,32574,20,46.0,90
9,content_c27558df2b0c,client_19581e27de,down,1240,104,4.9,257


## 5. Why ML beats a fixed rule here

ML beats a fixed rule here because "should this page be reviewed" depends on many signals interacting, not one clean threshold. The lane guide's own baseline rules (stale_visible_page, declining_with_demand, page_one_decay_risk, etc.) are each simple if-statements, and stacked together they only reach Precision@50 = 0.24. A page can be borderline on every single rule individually -- moderate impressions, moderate staleness, moderate position -- and still be
a real decline risk when those weak signals combine. A hand-written if-statement can't easily capture "several medium signals together matter more than any one strong signal," but a model that learns nonlinear combinations can -- which is exactly why the random forest jumps to 0.74. That measured 3x lift is itself the evidence that the pattern is real but too tangled for a fixed rule.

In [6]:
# Show how a single-threshold rule misses combined weak signals
close_calls = df[(df['impressions_90d'].between(80, 150)) &
                  (df['days_since_last_update'].between(120, 200)) &
                  (df['avg_position'].between(15, 30))]
print(f"Pages that are 'medium risk' on 3 signals at once (none individually crosses a hard threshold): {len(close_calls)}")
print(f"Of these, {(close_calls['trend_direction']=='down').mean():.1%} are actually declining -- a fixed single-column rule would likely miss most of these.")

Pages that are 'medium risk' on 3 signals at once (none individually crosses a hard threshold): 1
Of these, 0.0% are actually declining -- a fixed single-column rule would likely miss most of these.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.